In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import seaborn as sns

In [ ]:
DATAFOLDER = Path("../data")

ttl_file = DATAFOLDER / "ttls.csv"
file_with_preds = DATAFOLDER / "PB_NAapp-221024_PB62-221024-120029_Cam1.csv"

In [ ]:
ttls = pd.read_csv(ttl_file)
data = pd.read_csv(file_with_preds)
prob_app = data.Probability_Appetitive

In [ ]:
stub = 'PB62-221024-120029'

snips = []
for inf in ttls[stub].values:
    frame_on = int(inf * 10)
    
    snips.append(prob_app[frame_on-50:frame_on+150].values)

snips = np.array(snips[:-1])
    

In [ ]:
f, ax = plt.subplots(nrows=2, figsize=(10, 5), sharex=True)
sns.heatmap(snips, cmap="viridis", ax=ax[0])

ax[1].plot(np.mean(snips, axis=0))
ax[1].set_xlabel("Time")
ax[1].set_ylabel("Mean Probability")

sns.despine(ax=ax[1])

In [ ]:
# np.random.seed(0)
shifted_means = []

shifted=prob_app.copy()
for x in range(1000):
    shifted = np.roll(shifted, 300)

    snips = []
    for inf in ttls[stub].values:
        frame_on = int(inf * 10)
        
        snips.append(shifted[frame_on-50:frame_on+150])

    snips = np.array(snips[:-1])
    shifted_means.append(np.mean(snips, axis=0))

shifted_means = np.array(shifted_means)

real_snips = []
for inf in ttls[stub].values:
    frame_on = int(inf * 10)
    
    real_snips.append(prob_app[frame_on-50:frame_on+150].values)

real_snips = np.array(real_snips[:-1])

f, ax = plt.subplots(figsize=(5, 3))
ax.plot(np.mean(shifted_means, axis=0), label="Mean of shuffles")
ax.fill_between(range(200), np.percentile(shifted_means, 2.5, axis=0), np.percentile(shifted_means, 97.5, axis=0), alpha=0.3, label="95% CI")

ax.plot(np.mean(real_snips, axis=0), color="red", label="Real probability")

ax.axvline(50, color="black", linestyle="--")
ax.axvline(150, color="black", linestyle="--")
ax.set_xticks([0, 50, 100, 150, 200], labels=["-5s", "0s", "+5s", "+10s", "+15s"])

ax.set_ylabel("Prob of appetitive behaviour")

ax.legend()

sns.despine(ax=ax)


    

In [ ]:
real_snips_baselined = np.subtract(real_snips, np.mean(shifted_means))

plt.plot(np.mean(real_snips_baselined, axis=0))

sns.heatmap(real_snips_baselined, cmap="viridis")

In [ ]:
np.percentile(shifted_means, 97.5)

In [ ]:
plt.plot([np.sum(snip[50:150] > np.percentile(shifted_means, 97.5)) for snip in real_snips])